## Example: Dask `persist` with Sentinel-2 (remote cluster)

Based on Notebook 4’s real-data + Gateway setup, this notebook focuses on **`persist`**:

- **`compute()`** — run the graph and bring the result **to the client** (usually as NumPy / in-memory xarray).
- **`persist()`** — run the graph and keep the result **on the workers** as Futures. The object stays a lazy Dask collection, but chunks are already computed in cluster memory.

**When to use `persist`:** you will reuse the same intermediate result several times (stats, plots, further lazy ops, export) and want to avoid recomputing the expensive upstream graph.

In [ ]:
import datacube
from dask_gateway import Gateway
from dask.distributed import Client, wait
import matplotlib.pyplot as plt
import time

In [ ]:
# Initialise datacube

dc = datacube.Datacube()

In [ ]:
# (Central NSW)
x_min, x_max = 1200000, 1300000  # 100km wide
y_min, y_max = -3600000, -3700000  # 100km high
date_range = ("2024-01-01", "2024-02-28")

### Load data (lazy)

In [ ]:
product = "ga_s2bm_ard_3"  # Sentinel-2 B
measurements = ["nbart_red", "nbart_blue", "oa_s2cloudless_mask"]
output_crs = "EPSG:3577"
resolution = [-30, 30]

dask_chunks = {
    "time": 1,
    "y": 500,
    "x": 500
}

ds = dc.load(product=product,
             measurements=measurements,
             crs="EPSG:3577",
             x=(x_min, x_max),
             y=(y_min, y_max),
             time=date_range,
             output_crs=output_crs,
             resolution=resolution,
             dask_chunks=dask_chunks,
             dataset_predicate=lambda ds: ds.metadata.dataset_maturity == "final",
             skip_broken_datasets=True  # Important!
             )
ds

### Define a lazy computation

Nothing has run yet — this only builds a Dask graph.

In [ ]:
no_clouds_ds = ds.where(ds["oa_s2cloudless_mask"] == 1)
ratio_ds = no_clouds_ds["nbart_red"] / no_clouds_ds["nbart_blue"]
mean_ratio_ds = ratio_ds.mean(dim="time", skipna=True)
mean_ratio_ds

### Start a remote Dask cluster

In [ ]:
gateway = Gateway()

print(gateway.list_clusters())

options = gateway.cluster_options()
options.worker_cores = 1
options.worker_threads = 1
options.worker_memory = 1  # (GB)

cluster = gateway.new_cluster(cluster_options=options)

num_workers = 16
cluster.scale(num_workers)  # or .adapt(minimum=4, maximum=16)

client = Client(cluster)
print(client.dashboard_link)

client.wait_for_workers(n_workers=num_workers)

### Without `persist`: repeated work recomputes the graph

Each `.compute()` below starts from the original lazy load + mask + ratio + mean.
Watch the dashboard — the expensive upstream tasks run **twice**.

In [ ]:
%%time
min_val = mean_ratio_ds.min().compute()
print("min:", float(min_val))

In [ ]:
%%time
max_val = mean_ratio_ds.max().compute()
print("max:", float(max_val))

### With `persist`: compute once, keep on the workers

`persist()` submits the graph and leaves chunk results in **worker memory**.
Use `wait(...)` if you want to block until that materialisation finishes before timing downstream work.

In [ ]:
%%time
# Compute the mean-ratio once and hold it on the cluster
mean_ratio_persisted = mean_ratio_ds.persist()

# Optional but useful for demos/timings: wait until persist has finished
wait(mean_ratio_persisted)

mean_ratio_persisted

Downstream ops now start from the **persisted** chunks (no full reload / remask / re-mean).

In [ ]:
%%time
min_val = mean_ratio_persisted.min().compute()
print("min:", float(min_val))

In [ ]:
%%time
max_val = mean_ratio_persisted.max().compute()
print("max:", float(max_val))

### `persist` vs `compute` (same data)

| | **`persist()`** | **`compute()`** |
|---|---|---|
| Where results live | Workers (cluster memory) | Client process |
| Return type | Still a Dask-backed collection | Concrete NumPy / xarray |
| Good for | Reuse on cluster, further lazy ops, large arrays | Final small results, local plotting after shrink |
| Memory pressure | Workers must hold the chunks | Client must hold the full array |

Notebook 4 called `mean_ratio_ds.compute()` once and plotted locally.
Here we persist first (reuse on cluster), then pull a concrete copy only when needed for plotting.

In [ ]:
%%time
# Bring a concrete copy to the client for local plotting (as in Notebook 4)
mean_ratio_local = mean_ratio_persisted.compute()
mean_ratio_local

In [ ]:
# Visualise mean ratio dataset

band = mean_ratio_local

band.plot.imshow(cmap="viridis")
plt.title("Result (after persist)")
plt.xlabel("x")
plt.ylabel("y")
plt.show()

### Clean up

Persisted data is released when the cluster shuts down (or when you delete references and the scheduler GC’s the futures).

In [ ]:
client.close()
cluster.close(shutdown=True)

### Takeaways

1. Lazy graphs are recomputed from scratch on every `compute()` unless you **`persist`** (or otherwise cache) intermediates.
2. Use **`persist`** when the same expensive result feeds multiple later steps.
3. Use **`compute`** when you need the final values on the client (or the result is small).
4. Persisted data costs **worker RAM** — persist intermediates you reuse, not every temporary expression.
5. `wait(persisted)` is helpful when you want a clear “materialised” barrier before timing or dependent work.